# COS Method for Option Pricing — Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ee2625/fourier-cosine-option-pricing/blob/main/notebooks/demo.ipynb)

This notebook is a short, hands-on tour of the [`cos_pricing`](https://github.com/ee2625/fourier-cosine-option-pricing) package. The core idea: the **COS method** (Fang & Oosterlee, 2008) prices European options by expanding the risk-neutral log-price density as a finite cosine series on a truncated interval $[a, b]$. The cosine coefficients are read directly from the model's **characteristic function**

$$\varphi(u) = \mathbb{E}[e^{i u X_T}],$$

and the payoff is integrated *analytically*, giving exponential convergence in $N$ for smooth densities. Because we only ever need the characteristic function — not the density — COS handles models where the density is unknown in closed form, including:

- **Black-Scholes (BSM)** — sanity check; we know the analytical price.
- **Heston** — stochastic volatility, generates smiles and skew.
- **Variance Gamma (VG)** — pure-jump Lévy model with fat tails.
- **CGMY** — four-parameter pure-jump Lévy model.

**What this notebook shows:**
1. Price a BSM call with COS and check the error against the closed-form formula.
2. Plot exponential convergence: $|\text{err}|$ vs $N$.
3. Generate a Heston implied-volatility smile.
4. Compare Lévy-process smiles (VG vs CGMY).

For the full validation suite — paper-replication tables, Bermudan/American convergence, control-variate experiments — see `notebooks/tests.ipynb` in the repo.

## 1. Install the package

Run this cell first. It pulls the package straight from GitHub into the Colab runtime so the rest of the notebook works with no local setup.

In [ ]:
!pip install git+https://github.com/ee2625/fourier-cosine-option-pricing.git --quiet

## 2. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cos_pricing import (
    bsm_price,        # analytical Black-Scholes (reference)
    bsm_impvol,       # Black-Scholes implied vol inversion
    BsmModel,         # BSM under COS
    HestonCOSPricer,  # Heston under COS
    VgModel,          # Variance Gamma under COS
    CgmyModel,        # CGMY under COS
)

plt.rcParams["figure.dpi"] = 110
print("imports OK")

## 3. Simple worked example — BSM call, COS vs analytical

We price a European call under Black-Scholes two ways and compare. The COS price should match the closed-form Black-Scholes price to near machine precision once $N$ is large enough.

In [ ]:
# Market and contract parameters
S, K, T = 100.0, 100.0, 1.0
sigma, r, q = 0.25, 0.05, 0.0

# Analytical Black-Scholes price (reference)
ref = float(bsm_price(K, S, sigma, T, intr=r, divr=q, cp=+1))

# COS price with N = 128 cosine terms
bsm = BsmModel(sigma=sigma, intr=r, divr=q)
cos_128 = float(bsm.price(K, S, T, cp=+1, n_cos=128))

print(f"Analytical BSM:    {ref:.14f}")
print(f"COS price (N=128): {cos_128:.14f}")
print(f"Absolute error:    {abs(cos_128 - ref):.2e}")

## 4. Visualisation — exponential convergence

Sweep $N$ and plot $|\text{COS} - \text{analytical}|$ on a log scale. For a smooth density like BSM's lognormal, the error drops geometrically until it hits the double-precision floor (~$10^{-13}$ at this price scale).

In [ ]:
N_values = [8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256]
errors = np.array([abs(float(bsm.price(K, S, T, cp=+1, n_cos=N)) - ref) for N in N_values])

# Floor tiny errors at price-scale machine epsilon so the log plot stays readable
floor = 100.0 * np.finfo(float).eps   # ~2.22e-14
errors_plot = np.maximum(errors, floor)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.semilogy(N_values, errors_plot, "o-", color="#1f77b4")
ax.axhline(floor, color="#888", linestyle="--", linewidth=1.0,
           label=f"price-scale double-precision floor (~{floor:.0e})")
ax.set(xlabel="N (number of cosine terms)",
       ylabel="|COS price − analytical BSM|",
       title="COS convergence for a BSM call")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 5. Advanced example — Heston implied-volatility smile

Now the model has no closed-form density, only a characteristic function. We price a strip of strikes under Heston, then invert each price through the Black-Scholes formula to recover the **implied volatility**. The classic downward skew (puts pricier than calls at equal moneyness) appears, driven by the negative spot-vol correlation $\rho$.

Parameters are from Fang & Oosterlee (2008) Table 4. The ATM call (K=100) should come out to roughly 5.785, matching the paper's reference value 5.785155435.

In [ ]:
# Heston parameters: Fang & Oosterlee (2008), Table 4
heston = HestonCOSPricer(
    S0=100.0,
    v0=0.0175,    # initial variance
    lam=1.5768,   # mean-reversion speed (kappa in some texts)
    eta=0.5751,   # vol-of-vol
    ubar=0.0398,  # long-run variance (theta in some texts)
    rho=-0.5711,  # spot-vol correlation
    r=0.0, q=0.0,
)

T_smile = 1.0
strikes = np.linspace(70.0, 130.0, 25)

# Heston call prices via COS
prices_heston = np.array([
    heston.price_call(float(K), T_smile, N=160) for K in strikes
])

# Invert each price to a Black-Scholes implied vol
ivs_heston = np.array([
    float(bsm_impvol(p, float(K), 100.0, T_smile, intr=0.0, divr=0.0, cp=+1))
    for K, p in zip(strikes, prices_heston)
])

atm_price = float(heston.price_call(100.0, T_smile, N=160))
print(f"Heston ATM call (K=100, T=1):  {atm_price:.9f}")
print(f"F&O 2008 Table 4 reference:    5.785155435")
print(f"Difference:                    {abs(atm_price - 5.785155435):.2e}")

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(strikes, ivs_heston * 100.0, "o-", color="#d62728")
ax.axvline(100.0, color="#888", linestyle=":", linewidth=1.0, label="ATM (K = $S_0$)")
ax.set(xlabel="Strike K",
       ylabel="Black-Scholes implied vol (%)",
       title=f"Heston implied-vol smile, T = {T_smile}")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 6. Parameter sweep — Lévy-process smiles (VG vs CGMY)

Same idea, different families: pure-jump Lévy processes. Variance Gamma has three parameters $(\sigma, \theta, \nu)$; CGMY has four $(C, G, M, Y)$ that independently control jump activity and tail decay. Both produce non-trivial smiles even though there's no stochastic-volatility component — the smile comes entirely from the jump structure.

In [ ]:
T_levy = 1.0
strikes_levy = np.linspace(80.0, 120.0, 21)
r_levy = 0.1

# Variance Gamma
vg = VgModel(sigma=0.12, theta=-0.14, nu=0.2, intr=r_levy, divr=0.0)
prices_vg = np.array([
    float(vg.price(float(K), 100.0, T_levy, cp=+1, n_cos=256))
    for K in strikes_levy
])
ivs_vg = np.array([
    float(bsm_impvol(p, float(K), 100.0, T_levy, intr=r_levy, divr=0.0, cp=+1))
    for K, p in zip(strikes_levy, prices_vg)
])

# CGMY — Y < 1 → finite-variation jumps
cgmy = CgmyModel(C=1.0, G=5.0, M=10.0, Y=0.5, intr=r_levy, divr=0.0)
prices_cgmy = np.array([
    float(cgmy.price(float(K), 100.0, T_levy, cp=+1, n_cos=256))
    for K in strikes_levy
])
ivs_cgmy = np.array([
    float(bsm_impvol(p, float(K), 100.0, T_levy, intr=r_levy, divr=0.0, cp=+1))
    for K, p in zip(strikes_levy, prices_cgmy)
])

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(strikes_levy, ivs_vg * 100.0, "o-", color="#1f77b4",
        label="Variance Gamma  (σ=0.12, θ=-0.14, ν=0.2)")
ax.plot(strikes_levy, ivs_cgmy * 100.0, "s-", color="#2ca02c",
        label="CGMY  (C=1, G=5, M=10, Y=0.5)")
ax.axvline(100.0, color="#888", linestyle=":", linewidth=1.0)
ax.set(xlabel="Strike K",
       ylabel="Black-Scholes implied vol (%)",
       title=f"Lévy-process smiles, T = {T_levy}")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## What to read next

- `notebooks/tests.ipynb` — full paper-replication suite (F&O 2008 Tables 1–10, F&O 2009 Bermudan/American limits, plus our dimensionless-invariance work).
- `src/cos_pricing/` — the package source. `BsmModel`, `HestonCOSPricer`, `VgModel`, `CgmyModel`, plus `carr_madan_price`, `lewis_price`, `frft_price` for cross-method comparisons.
- `tests/` — automated unit and randomized property tests.

**References:**
- Fang, F. & Oosterlee, C. W. (2008). *A novel pricing method for European options based on Fourier-cosine series expansions*. SIAM J. Sci. Comput., 31(2), 826–848.
- Fang, F. & Oosterlee, C. W. (2009). *Pricing early-exercise and discrete barrier options by Fourier-cosine series expansions*. Numer. Math., 114(1), 27–62.